In [5]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from weather import get_weather

def fetch_all_cities(cities):
    """Fetch current weather for every city in `cities`.
    Args:
        cities: A list of dicts, each with the keys "name", "lat" and "lon".
    Returns:
        A list of result dicts, one per city that was fetched successfully,
        each of the form:
            {"city": <name>,
             "fetched_at": <utc timestamp>,
             "weather": <dict returned by get_weather()>
             }
        Cities whose fetch fails (get_weather returns None) are skipped, so
        no None values ever reach the log.
    """
    # Store the formatted weather record created for each city.
    results = []
    for city in cities:
        # Use the city's coordinates to request its current weather data.
        weather = get_weather(city["lat"], city["lon"])
        if weather is None:
            print(f"Skipping {city['name']}: no weather data returned.")
            continue
        # Combine the city name, current UTC timestamp, and weather response.
        results.append(
            {
            "city": city["name"],
            "fetched_at": datetime.now(timezone.utc).isoformat(),
            "weather": weather
            })
    # Return the complete batch after every city has been processed.
    return results

In [6]:
def update_log(log_path, new_results):
    """Append `new_results` to the JSON log at `log_path`.
    Loads the existing log (an empty list if the file does not exist yet),
    appends the new entries, and writes the combined list back. The file is
    always a JSON array of result dicts.
    Args:
        log_path:    Path of the JSON log file.
        new_results: List of result dicts from fetch_all_cities().
    Returns:
        None.
    """

    path = Path(log_path)
    # Load the existing weather history, or start a new log if the file is absent.
    if path.exists():
        with path.open("r", encoding="utf-8") as file:
            log = json.load(file)
    else:
        log = []

    # Add every result from the latest collection batch to the history.
    log.extend(new_results)

    # Save the complete updated history as an easy-to-read JSON array.
    with path.open("w", encoding="utf-8") as file:
        json.dump(log, file, indent=2)

In [7]:
def summarise_log(log_path):
    """Compute summary statistics for the JSON log at `log_path`.
    Args:
        log_path: Path of the JSON log file.
    Returns:
        A dict with four keys:
            total_records:   Number of entries in the log.
            cities_tracked:  Sorted list of unique city names.
            latest_fetch:    Most recent "fetched_at" value, or None if empty.
            avg_temperature: Mean temperature to two decimal places, or None
                             if empty.
    """

    path = Path(log_path)
    # Load the existing log, or use an empty list if no log has been created yet.
    if path.exists():
        with path.open("r", encoding="utf-8") as file:
            log = json.load(file)
    else:
        log = []

    # Avoid calculations that require records when the log is empty.
    if not log:
        return {
            "total_records": 0,
            "cities_tracked": [],
            "latest_fetch": None,
            "avg_temperature": None,
        }

    # Extract every recorded temperature so their mean can be calculated.
    temperatures = [entry["weather"]["temperature"] for entry in log]

    # UTC timestamps can be compared directly to find the latest one.
    return {
        "total_records": len(log),
        "cities_tracked": sorted({entry["city"] for entry in log}),
        "latest_fetch": max(entry["fetched_at"] for entry in log),
        "avg_temperature": round(sum(temperatures) / len(temperatures), 2),
    }
